<a href="https://colab.research.google.com/github/yashrohilla25/cudalab4/blob/main/parallellab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Problem 1 1024 threads
%%writefile problem1.cu
#include <iostream>

#define N 1024

__global__ void task_kernel(int *input, int *output) {
    int tid = threadIdx.x;

    if (tid == 0) {
        // Task a: iterative sum
        int sum = 0;
        for (int i = 0; i < N; i++) {
            sum += input[i];
        }
        output[0] = sum;
    }

    if (tid == 1) {
        // Task b: formula-based sum
        output[1] = (N * (N + 1)) / 2;
    }
}

int main() {
    int *h_input, *h_output;
    int *d_input, *d_output;

    size_t size = N * sizeof(int);
    h_input = (int*)malloc(size);
    h_output = (int*)malloc(2 * sizeof(int));  // for 2 results

    for (int i = 0; i < N; ++i) {
        h_input[i] = i + 1;
    }

    cudaMalloc((void**)&d_input, size);
    cudaMalloc((void**)&d_output, 2 * sizeof(int));

    cudaMemcpy(d_input, h_input, size, cudaMemcpyHostToDevice);

    // Launch with at least 2 threads
    task_kernel<<<1, 2>>>(d_input, d_output);

    cudaMemcpy(h_output, d_output, 2 * sizeof(int), cudaMemcpyDeviceToHost);

    std::cout << "Iterative sum: " << h_output[0] << std::endl;
    std::cout << "Formula sum: " << h_output[1] << std::endl;

    cudaFree(d_input);
    cudaFree(d_output);
    free(h_input);
    free(h_output);

    return 0;
}


Overwriting problem1.cu


In [ ]:
!nvcc problem1.cu -o problem1  -arch=sm_75
!./problem1


Iterative sum: 524800
Formula sum: 524800


In [ ]:
#Q2(A) in cpp
# Write the C++ code to a file
code = """
#include <iostream>
#include <omp.h>
#include <chrono>
#include <cstdlib>

#define N 1000

void merge(int arr[], int l, int m, int r) {
    int i, j, k;
    int n1 = m - l + 1;
    int n2 = r - m;

    int *L = new int[n1];
    int *R = new int[n2];

    for (i = 0; i < n1; i++) L[i] = arr[l + i];
    for (j = 0; j < n2; j++) R[j] = arr[m + 1 + j];

    i = 0; j = 0; k = l;
    while (i < n1 && j < n2)
        arr[k++] = (L[i] <= R[j]) ? L[i++] : R[j++];

    while (i < n1) arr[k++] = L[i++];
    while (j < n2) arr[k++] = R[j++];

    delete[] L;
    delete[] R;
}

void mergeSort(int arr[], int l, int r) {
    if (l < r) {
        int m = l + (r - l) / 2;
        #pragma omp parallel sections
        {
            #pragma omp section
            mergeSort(arr, l, m);

            #pragma omp section
            mergeSort(arr, m + 1, r);
        }
        merge(arr, l, m, r);
    }
}

int main() {
    int arr[N];
    for (int i = 0; i < N; i++)
        arr[i] = rand() % 1000;

    auto start = std::chrono::high_resolution_clock::now();
    mergeSort(arr, 0, N - 1);
    auto stop = std::chrono::high_resolution_clock::now();

    auto duration = std::chrono::duration_cast<std::chrono::microseconds>(stop - start);
    std::cout << "Time taken (Pipelining - OpenMP): " << duration.count() << " microseconds\\n";
    return 0;
}
"""

with open("mergesort_pipelining.cpp", "w") as f:
    f.write(code)

# Compile and run
!g++ -fopenmp mergesort_pipelining.cpp -o pipelining && ./pipelining


Time taken (Pipelining - OpenMP): 790 microseconds


In [ ]:
#Q2 (b) CUDA Runtime
%%writefile cudamergesort.cu
#include <iostream>
#include <chrono>
#define N 1000

__global__ void mergeSortKernel(int *d_data) {
    int tid = threadIdx.x;

    for (int size = 2; size <= N; size *= 2) {
        int step = size / 2;
        if (tid % size == 0) {
            int start = tid;
            int mid = start + step;
            int end = min(start + size, N);

            int *temp = new int[size];
            int i = start, j = mid, k = 0;

            while (i < mid && j < end)
                temp[k++] = (d_data[i] < d_data[j]) ? d_data[i++] : d_data[j++];
            while (i < mid) temp[k++] = d_data[i++];
            while (j < end) temp[k++] = d_data[j++];

            for (i = 0; i < k; i++)
                d_data[start + i] = temp[i];

            delete[] temp;
        }
        __syncthreads();
    }
}

int main() {
    int *h_data = new int[N];
    for (int i = 0; i < N; i++)
        h_data[i] = rand() % 1000;

    int *d_data;
    cudaMalloc((void**)&d_data, N * sizeof(int));
    cudaMemcpy(d_data, h_data, N * sizeof(int), cudaMemcpyHostToDevice);

    auto start = std::chrono::high_resolution_clock::now();
    mergeSortKernel<<<1, N>>>(d_data);
    cudaDeviceSynchronize();
    auto stop = std::chrono::high_resolution_clock::now();

    cudaMemcpy(h_data, d_data, N * sizeof(int), cudaMemcpyDeviceToHost);
    auto duration = std::chrono::duration_cast<std::chrono::microseconds>(stop - start);

    std::cout << "Time taken (CUDA Merge Sort): " << duration.count() << " microseconds\n";

    cudaFree(d_data);
    delete[] h_data;
    return 0;
}


Writing cudamergesort.cu


In [ ]:
!nvcc cudamergesort.cu -o cudamergesort-arch=sm_75
!./cudamergesort


Time taken (CUDA Merge Sort): 3724 microseconds
